# 02 — Arcade : alternative moderne

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- créer une fenêtre Arcade avec héritage de `arcade.Window` ;
- dessiner des formes et afficher du texte ;
- gérer les sprites avec `SpriteList` ;
- gérer les événements clavier et souris ;
- détecter les collisions entre sprites ;
- organiser le jeu avec des Vues (menus, écrans) ;
- jouer des sons.

## Prérequis — ce que vous devez maîtriser

À ce stade de la formation, vous savez :

- Python, classes, héritage ;
- les bases des boucles de jeu.

Ce que ce notebook **ne couvrira pas** :

- le rendu GPU bas niveau (OpenGL) ;
- les moteurs professionnels (Unity, Godot).

## Plan

1. Installation
2. Fenêtre de base
3. Dessin de formes
4. Texte
5. Sprites et SpriteList
6. Mouvement et delta time
7. Événements clavier
8. Événements souris
9. Collisions
10. Sons
11. Vues (scènes)
12. Synthèse
13. Exercices
14. Ressources externes

### Exemple avancé : plateforme 2D avec gravité

Arcade simplifie énormément la création de jeux de plateforme
grâce à son moteur physique intégré.

In [ ]:
import arcade

class PlatformerGame(arcade.Window):
    def __init__(self):
        super().__init__(800, 600, 'Mini Platformer')
        arcade.set_background_color(arcade.color.LIGHT_BLUE)

        # Joueur
        self.player = arcade.SpriteSolidColor(30, 30, arcade.color.RED)
        self.player.center_x = 100
        self.player.center_y = 200

        # Plateformes
        self.platforms = arcade.SpriteList()
        # Sol
        for x in range(0, 801, 64):
            wall = arcade.SpriteSolidColor(64, 20, arcade.color.DARK_GREEN)
            wall.center_x = x
            wall.center_y = 32
            self.platforms.append(wall)
        # Plateformes flottantes
        for (px, py) in [(200, 150), (400, 250), (600, 350)]:
            plat = arcade.SpriteSolidColor(120, 15, arcade.color.BROWN)
            plat.center_x = px
            plat.center_y = py
            self.platforms.append(plat)

        # Physique
        self.physics = arcade.PhysicsEnginePlatformer(
            self.player, self.platforms, gravity_constant=0.8
        )

    def on_key_press(self, key, mods):
        if key == arcade.key.LEFT:
            self.player.change_x = -5
        elif key == arcade.key.RIGHT:
            self.player.change_x = 5
        elif key == arcade.key.UP and self.physics.can_jump():
            self.player.change_y = 15

    def on_key_release(self, key, mods):
        if key in (arcade.key.LEFT, arcade.key.RIGHT):
            self.player.change_x = 0

    def on_update(self, delta_time):
        self.physics.update()

    def on_draw(self):
        self.clear()
        self.platforms.draw()
        self.player.draw()

# Décommenter pour jouer :
# game = PlatformerGame()
# arcade.run()
print('Jeu de plateforme défini')

### Exercice : collecte d'objets

Ajoutez au jeu de plateforme :
1. Des pièces (cercles jaunes) placées sur les plateformes
2. Un score qui augmente quand le joueur touche une pièce
3. Un affichage HUD (score + nombre de pièces restantes)
4. Un message "Victoire !" quand toutes les pièces sont collectées

In [ ]:
import arcade

class CollectGame(arcade.Window):
    def __init__(self):
        super().__init__(800, 600, 'Collecte')
        arcade.set_background_color(arcade.color.SKY_BLUE)

        self.player = arcade.SpriteSolidColor(25, 25, arcade.color.BLUE)
        self.player.center_x = 100
        self.player.center_y = 200

        # Coins
        self.coins = arcade.SpriteList()
        for (cx, cy) in [(200, 180), (400, 280), (600, 380), (300, 100), (500, 200)]:
            coin = arcade.SpriteCircle(10, arcade.color.GOLD)
            coin.center_x = cx
            coin.center_y = cy
            self.coins.append(coin)

        # Sol
        self.walls = arcade.SpriteList()
        for x in range(0, 801, 64):
            w = arcade.SpriteSolidColor(64, 20, arcade.color.DARK_GREEN)
            w.center_x = x; w.center_y = 32
            self.walls.append(w)

        self.physics = arcade.PhysicsEnginePlatformer(
            self.player, self.walls, gravity_constant=0.8
        )
        self.score = 0

    def on_key_press(self, key, mods):
        if key == arcade.key.LEFT: self.player.change_x = -5
        elif key == arcade.key.RIGHT: self.player.change_x = 5
        elif key == arcade.key.UP and self.physics.can_jump():
            self.player.change_y = 14

    def on_key_release(self, key, mods):
        if key in (arcade.key.LEFT, arcade.key.RIGHT):
            self.player.change_x = 0

    def on_update(self, delta_time):
        self.physics.update()
        hits = arcade.check_for_collision_with_list(self.player, self.coins)
        for coin in hits:
            coin.remove_from_sprite_lists()
            self.score += 10

    def on_draw(self):
        self.clear()
        self.walls.draw()
        self.coins.draw()
        self.player.draw()
        arcade.draw_text(f'Score: {self.score}', 10, 570, arcade.color.WHITE, 20)
        if len(self.coins) == 0:
            arcade.draw_text('VICTOIRE !', 300, 300, arcade.color.GOLD, 40)

# game = CollectGame()
# arcade.run()
print('Jeu de collecte défini')

### Exercice avancé : tir de projectiles

Ajoutez un mécanisme de tir :
1. Appuyer sur ESPACE crée un projectile qui va vers la droite
2. Les projectiles disparaissent en sortant de l'écran
3. Ajoutez des cibles qui se détruisent au contact

In [ ]:
import arcade
import math

class Bullet(arcade.SpriteSolidColor):
    def __init__(self, x, y, direction=1):
        super().__init__(8, 4, arcade.color.YELLOW)
        self.center_x = x
        self.center_y = y
        self.change_x = 8 * direction

    def update(self):
        self.center_x += self.change_x
        if self.center_x > 850 or self.center_x < -50:
            self.remove_from_sprite_lists()

class Target(arcade.SpriteSolidColor):
    def __init__(self, x, y):
        super().__init__(20, 20, arcade.color.RED)
        self.center_x = x
        self.center_y = y
        self.change_y = 1
        self.base_y = y

    def update(self):
        self.center_y += self.change_y
        if abs(self.center_y - self.base_y) > 30:
            self.change_y *= -1

# Intégration dans la boucle de jeu :
# bullets = arcade.SpriteList()
# targets = arcade.SpriteList()
# Dans on_key_press(SPACE): bullets.append(Bullet(player.x, player.y))
# Dans on_update: vérifier collisions bullets vs targets
print('Système de tir défini')

### Comparaison Pygame-CE vs Arcade

| Critère | Pygame-CE | Arcade |
|---------|-----------|--------|
| Niveau | Bas niveau | Haut niveau |
| Boucle de jeu | Manuelle | Méthodes de classe |
| Physique | Manuelle | Moteur intégré |
| Rendu | SDL2 (CPU) | OpenGL (GPU) |
| Sprites | `Group` basique | `SpriteList` optimisée |
| Tilemaps | Non natif | Support TMX |
| Courbe d'apprentissage | Progressive | Plus simple |

**Conseil** : Pygame-CE pour comprendre les fondamentaux,
Arcade pour prototyper rapidement.

---

## 1. Installation

In [ ]:
# pip install arcade
import arcade
print(f"Arcade installé")

---

## 2. Fenêtre de base

On hérite de `arcade.Window` et on surcharge les méthodes.

In [ ]:
class MyGame(arcade.Window):
    def __init__(self) -> None:
        super().__init__(800, 600, 'Mon jeu')
        self.background_color = arcade.color.DARK_BLUE

    def on_draw(self) -> None:
        self.clear()

print('MyGame défini (appeler MyGame().run() hors notebook)')

### Différences avec Pygame

- Arcade est **orienté objet** par design (héritage).
- Les coordonnées ont l’**origine en bas-gauche** (comme en maths).
- La boucle est gérée automatiquement par `run()`.

---

## 3. Dessin de formes

In [ ]:
# Dans on_draw() :
# arcade.draw_rectangle_filled(x, y, w, h, color)
# arcade.draw_circle_filled(x, y, radius, color)
# arcade.draw_line(x1, y1, x2, y2, color, line_width)
# arcade.draw_polygon_filled(points, color)
print('draw_rectangle_filled, draw_circle_filled, draw_line, draw_polygon_filled')

---

## 4. Texte

In [ ]:
# arcade.draw_text('Score: 42', x, y, arcade.color.WHITE, font_size=20)
# arcade.draw_text('Centré', 400, 300, arcade.color.WHITE, 20,
#                  anchor_x='center', anchor_y='center')
print('draw_text avec ancrage')

---

## 5. Sprites et SpriteList

`SpriteList` utilise le batching GPU pour un rendu rapide.

In [ ]:
# player = arcade.Sprite('player.png', scale=0.5)
# player.center_x = 400
# player.center_y = 300
# ou avec SpriteSolidColor :
# player = arcade.SpriteSolidColor(40, 40, color=arcade.color.RED)
print('Sprite(path, scale=) ou SpriteSolidColor(w, h, color=)')

In [ ]:
# Liste de sprites (batching GPU)
# self.player_list = arcade.SpriteList()
# self.player_list.append(player)
# Dans on_draw :
# self.player_list.draw()
print('SpriteList : append(), draw(), update()')

---

## 6. Mouvement et delta time

In [ ]:
# def on_update(self, delta_time: float) -> None:
#     self.player.center_x += self.player.change_x * delta_time
#     self.player.center_y += self.player.change_y * delta_time
print('on_update(delta_time) appelé ~60 fois/s')

In [ ]:
# Ou laisser Arcade gérer :
# self.player.change_x = 5  # pixels par frame
# self.player_list.update()  # applique change_x/change_y
print('change_x / change_y + SpriteList.update()')

---

## 7. Événements clavier

In [ ]:
# def on_key_press(self, key: int, modifiers: int) -> None:
#     if key == arcade.key.LEFT:
#         self.player.change_x = -5
#     elif key == arcade.key.UP:
#         self.player.change_y = 5
#
# def on_key_release(self, key: int, modifiers: int) -> None:
#     if key in (arcade.key.LEFT, arcade.key.RIGHT):
#         self.player.change_x = 0
print('on_key_press / on_key_release')

---

## 8. Événements souris

In [ ]:
# def on_mouse_motion(self, x: int, y: int, dx: int, dy: int) -> None:
#     self.player.center_x = x
#     self.player.center_y = y
#
# def on_mouse_press(self, x, y, button, modifiers):
#     if button == arcade.MOUSE_BUTTON_LEFT:
#         print(f'Clic en ({x}, {y})')
print('on_mouse_motion, on_mouse_press, on_mouse_release')

---

## 9. Collisions

In [ ]:
# hits = arcade.check_for_collision_with_list(self.player, self.enemies)
# for enemy in hits:
#     enemy.remove_from_sprite_lists()
#     self.score += 1
print('check_for_collision_with_list -> liste de sprites touchés')

In [ ]:
# Collision entre deux sprites individuels
# if arcade.check_for_collision(sprite_a, sprite_b):
#     print('Collision !')
print('check_for_collision(sprite_a, sprite_b) -> bool')

---

## 10. Sons

In [ ]:
# son = arcade.load_sound(':resources:sounds/laser1.wav')
# arcade.play_sound(son, volume=0.5)
print('load_sound + play_sound')

---

## 11. Vues (scènes)

Les `View` permettent de gérer différents écrans (menu, jeu, game over).

In [ ]:
class MenuView(arcade.View):
    def on_draw(self) -> None:
        self.clear()
        arcade.draw_text('Appuyez sur ESPACE', 200, 300, arcade.color.WHITE, 30)

    def on_key_press(self, key: int, modifiers: int) -> None:
        if key == arcade.key.SPACE:
            game_view = GameView()  # créer la vue du jeu
            self.window.show_view(game_view)

print('View + show_view() pour changer d\'écran')

---

## Synthèse

| Élément | Arcade | Notes |
|---|---|---|
| Fenêtre | `class X(arcade.Window)` | + `run()` |
| Dessin | `draw_rectangle_filled`, `draw_circle_filled` | |
| Texte | `draw_text(text, x, y, color, size)` | |
| Sprite | `arcade.Sprite(path)` ou `SpriteSolidColor` | |
| SpriteList | `arcade.SpriteList()` + `draw()` | Batching GPU |
| Update | `def on_update(self, delta_time)` | |
| Clavier | `on_key_press` / `on_key_release` | |
| Souris | `on_mouse_motion` / `on_mouse_press` | |
| Collision | `check_for_collision_with_list` | |
| Sons | `load_sound` + `play_sound` | |
| Vues | `class X(arcade.View)` + `show_view()` | |

### Règles à retenir

- **Arcade est plus haut niveau que Pygame**, API orientée objet.
- **SpriteList** est indispensable pour les performances (batching GPU).
- **Views** pour organiser menus, jeu, game over.
- **Coordonnées** : origine en **bas-gauche** (comme en maths).
- **`on_update(delta_time)`** reçoit le temps en secondes.

---

---

## Physics Engine

Arcade inclut des moteurs physiques simples pour gerer les collisions avec les murs.

In [ ]:
# self.physics = arcade.PhysicsEngineSimple(self.player, self.walls)
# Dans on_update :
# self.physics.update()
print('PhysicsEngineSimple(moving_sprite, walls_list)')

In [ ]:
# Platformer engine (gravite + sauts)
# self.physics = arcade.PhysicsEnginePlatformer(
#     self.player, self.walls, gravity_constant=0.5
# )
# Dans on_key_press :
# if key == arcade.key.SPACE and self.physics.can_jump():
#     self.player.change_y = 10
print('PhysicsEnginePlatformer avec gravite et can_jump()')

---

## Tilemaps

In [ ]:
# Charger une tilemap (Tiled editor)
# map = arcade.load_tilemap('level.json', scaling=1)
# self.scene = arcade.Scene.from_tilemap(map)
# self.scene.draw()
print('load_tilemap + Scene.from_tilemap pour les niveaux')

---

## Camera

In [ ]:
# from arcade import Camera2D
# self.camera = Camera2D()
# Dans on_update :
# self.camera.position = self.player.position
# Dans on_draw :
# self.camera.use()
print('Camera2D pour suivre le joueur')

---

## Particules et effets

In [ ]:
# Effet basique avec des sprites temporaires
# class Particle(arcade.SpriteCircle):
#     def __init__(self, x, y):
#         super().__init__(3, arcade.color.YELLOW)
#         self.center_x = x
#         self.center_y = y
#         self.change_x = random.uniform(-2, 2)
#         self.change_y = random.uniform(1, 4)
#         self.lifetime = 30
#     def update(self):
#         self.center_x += self.change_x
#         self.center_y += self.change_y
#         self.lifetime -= 1
#         if self.lifetime <= 0:
#             self.remove_from_sprite_lists()
print('Particules = sprites avec duree de vie')

---

## Scheduling (actions differees)

In [ ]:
# arcade.schedule(callback, interval)
# arcade.unschedule(callback)
# Exemple : spawner un ennemi toutes les 3 secondes
# def spawn_enemy(delta_time):
#     e = arcade.SpriteSolidColor(20, 20, color=arcade.color.RED)
#     e.center_x = random.randint(50, 550)
#     e.center_y = 400
#     self.enemies.append(e)
# arcade.schedule(spawn_enemy, 3.0)
print('arcade.schedule(callback, interval_seconds)')

---

## Animations de sprites

In [ ]:
# Changer de texture a chaque frame
# class AnimatedSprite(arcade.Sprite):
#     def __init__(self):
#         super().__init__()
#         self.textures = [
#             arcade.load_texture(f'frame_{i}.png') for i in range(4)
#         ]
#         self.cur_texture = 0
#         self.texture = self.textures[0]
#     def update_animation(self, delta_time):
#         self.cur_texture = (self.cur_texture + 1) % len(self.textures)
#         self.texture = self.textures[self.cur_texture]
print('Changer self.texture dans update_animation()')

## Exercices

### Exercice 1 *(facile)* — Fenêtre basique

Créez une fenêtre Arcade 600x400 avec un fond vert.

In [ ]:
import arcade

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import arcade

class Game(arcade.Window):
    def __init__(self):
        super().__init__(600, 400, 'Demo')
        self.background_color = arcade.color.FOREST_GREEN
    def on_draw(self):
        self.clear()

# Game().run()
```

</details>

### Exercice 2 *(facile)* — Cercle qui suit la souris

Dessinez un cercle bleu de rayon 20 qui suit la position de la souris.

In [ ]:
import arcade

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import arcade

class Game(arcade.Window):
    def __init__(self):
        super().__init__(600, 400)
        self.mx, self.my = 300, 200
    def on_draw(self):
        self.clear()
        arcade.draw_circle_filled(self.mx, self.my, 20, arcade.color.BLUE)
    def on_mouse_motion(self, x, y, dx, dy):
        self.mx, self.my = x, y

# Game().run()
```

</details>

### Exercice 3 *(moyen)* — Sprite mobile

Créez un sprite rouge 40x40 qui se déplace avec les flèches du clavier.

In [ ]:
import arcade

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import arcade

class Game(arcade.Window):
    def __init__(self):
        super().__init__(600, 400)
        self.player = arcade.SpriteSolidColor(40, 40, color=arcade.color.RED)
        self.player.center_x, self.player.center_y = 300, 200
        self.sprites = arcade.SpriteList()
        self.sprites.append(self.player)
    def on_draw(self):
        self.clear()
        self.sprites.draw()
    def on_update(self, dt):
        self.player.center_x += self.player.change_x
        self.player.center_y += self.player.change_y
    def on_key_press(self, key, m):
        if key == arcade.key.LEFT: self.player.change_x = -5
        if key == arcade.key.RIGHT: self.player.change_x = 5
        if key == arcade.key.UP: self.player.change_y = 5
        if key == arcade.key.DOWN: self.player.change_y = -5
    def on_key_release(self, key, m):
        if key in (arcade.key.LEFT, arcade.key.RIGHT): self.player.change_x = 0
        if key in (arcade.key.UP, arcade.key.DOWN): self.player.change_y = 0

# Game().run()
```

</details>

### Exercice 4 *(moyen)* — Collisions

Ajoutez 5 ennemis bleus aléatoirement placés. Quand le joueur les touche, ils disparaissent.

In [ ]:
import arcade
import random

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import arcade
import random

class Game(arcade.Window):
    def __init__(self):
        super().__init__(600, 400)
        self.player = arcade.SpriteSolidColor(30, 30, color=arcade.color.RED)
        self.player.center_x, self.player.center_y = 300, 200
        self.players = arcade.SpriteList()
        self.players.append(self.player)
        self.enemies = arcade.SpriteList()
        for _ in range(5):
            e = arcade.SpriteSolidColor(20, 20, color=arcade.color.BLUE)
            e.center_x = random.randint(50, 550)
            e.center_y = random.randint(50, 350)
            self.enemies.append(e)
    def on_draw(self):
        self.clear()
        self.players.draw()
        self.enemies.draw()
    def on_update(self, dt):
        self.player.center_x += self.player.change_x
        self.player.center_y += self.player.change_y
        hits = arcade.check_for_collision_with_list(self.player, self.enemies)
        for e in hits:
            e.remove_from_sprite_lists()
    def on_key_press(self, key, m):
        if key == arcade.key.LEFT: self.player.change_x = -5
        if key == arcade.key.RIGHT: self.player.change_x = 5
        if key == arcade.key.UP: self.player.change_y = 5
        if key == arcade.key.DOWN: self.player.change_y = -5
    def on_key_release(self, key, m):
        self.player.change_x = 0
        self.player.change_y = 0

# Game().run()
```

</details>

### Exercice 5 *(difficile)* — Pong simplifié

Implémentez un Pong mono-joueur :
- Raquette en bas contrôlée par gauche/droite
- Balle qui rebondit sur les trois murs
- Si la balle passe sous la raquette, game over

In [ ]:
import arcade

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import arcade

class Pong(arcade.Window):
    def __init__(self):
        super().__init__(600, 400)
        self.paddle = arcade.SpriteSolidColor(80, 10, color=arcade.color.WHITE)
        self.paddle.center_x = 300
        self.paddle.center_y = 30
        self.ball = arcade.SpriteSolidColor(10, 10, color=arcade.color.YELLOW)
        self.ball.center_x = 300
        self.ball.center_y = 200
        self.bvx, self.bvy = 200, -200
        self.all = arcade.SpriteList()
        self.all.append(self.paddle)
        self.all.append(self.ball)
    def on_draw(self):
        self.clear()
        self.all.draw()
    def on_update(self, dt):
        self.paddle.center_x += self.paddle.change_x * dt
        self.ball.center_x += self.bvx * dt
        self.ball.center_y += self.bvy * dt
        if self.ball.left < 0 or self.ball.right > 600: self.bvx *= -1
        if self.ball.top > 400: self.bvy *= -1
        if arcade.check_for_collision(self.ball, self.paddle): self.bvy = abs(self.bvy)
        if self.ball.bottom < 0:
            arcade.draw_text('GAME OVER', 200, 200, arcade.color.RED, 30)
    def on_key_press(self, key, m):
        if key == arcade.key.LEFT: self.paddle.change_x = -300
        if key == arcade.key.RIGHT: self.paddle.change_x = 300
    def on_key_release(self, key, m):
        self.paddle.change_x = 0

# Pong().run()
```

</details>

### Approfondir : utilisation des Views

Arcade 2.6+ introduit le système de `View` pour gérer les écrans
(menu, jeu, pause, game over) de manière élégante.

In [ ]:
import arcade

class MenuView(arcade.View):
    def on_draw(self):
        self.clear()
        arcade.draw_text('MENU PRINCIPAL', 400, 350,
                        arcade.color.WHITE, 30, anchor_x='center')
        arcade.draw_text('Appuyez sur ENTREE pour jouer', 400, 280,
                        arcade.color.GRAY, 18, anchor_x='center')

    def on_key_press(self, key, mods):
        if key == arcade.key.RETURN:
            game_view = GameView()
            self.window.show_view(game_view)

class GameView(arcade.View):
    def __init__(self):
        super().__init__()
        self.score = 0
        self.player = arcade.SpriteSolidColor(25, 25, arcade.color.BLUE)
        self.player.center_x = 400
        self.player.center_y = 300

    def on_draw(self):
        self.clear()
        self.player.draw()
        arcade.draw_text(f'Score: {self.score}', 10, 570, arcade.color.WHITE, 16)

    def on_key_press(self, key, mods):
        if key == arcade.key.ESCAPE:
            self.window.show_view(MenuView())
        if key == arcade.key.LEFT: self.player.change_x = -5
        if key == arcade.key.RIGHT: self.player.change_x = 5

    def on_key_release(self, key, mods):
        if key in (arcade.key.LEFT, arcade.key.RIGHT):
            self.player.change_x = 0

    def on_update(self, dt):
        self.player.update()
        self.score += 1

# window = arcade.Window(800, 600, 'Views Demo')
# window.show_view(MenuView())
# arcade.run()
print('Système de Views défini')

### Exercice : effets visuels avec les particules Arcade

Arcade intègre un système de particules via `arcade.Emitter`.
Créez un effet de feu d'artifice déclenché au clic.

In [ ]:
import arcade
import random

class FireworkView(arcade.View):
    def __init__(self):
        super().__init__()
        self.emitters = []

    def on_mouse_press(self, x, y, button, mods):
        """Crée un feu d'artifice au clic."""
        colors = [
            arcade.color.RED, arcade.color.GOLD,
            arcade.color.CYAN, arcade.color.MAGENTA
        ]
        color = random.choice(colors)
        e = arcade.Emitter(
            center_xy=(x, y),
            emit_controller=arcade.EmitBurst(50),
            particle_factory=lambda emitter: arcade.FadeParticle(
                filename_or_texture=arcade.make_circle_texture(6, color),
                change_xy=(
                    random.uniform(-3, 3),
                    random.uniform(-3, 3)
                ),
                lifetime=random.uniform(0.5, 2.0),
            )
        )
        self.emitters.append(e)

    def on_update(self, dt):
        for e in self.emitters[:]:
            e.update()
            if e.can_reap():
                self.emitters.remove(e)

    def on_draw(self):
        self.clear()
        for e in self.emitters:
            e.draw()
        arcade.draw_text('Cliquez pour un feu d\'artifice !',
                        400, 20, arcade.color.WHITE, 16, anchor_x='center')

# window = arcade.Window(800, 600, 'Fireworks')
# window.show_view(FireworkView())
# arcade.run()
print('Feux d\'artifice définis')

### Exercice : scrolling camera

Pour les niveaux plus grands que l'écran, Arcade offre une caméra :
1. Créez un niveau de 3000x600 pixels
2. Placez des plateformes sur toute la largeur
3. La caméra suit le joueur horizontalement
4. Affichez un mini-map en haut de l'écran

In [ ]:
import arcade

class ScrollingGame(arcade.Window):
    def __init__(self):
        super().__init__(800, 600, 'Scrolling')
        self.camera = arcade.Camera2D()
        self.gui_camera = arcade.Camera2D()  # Pour le HUD

        self.player = arcade.SpriteSolidColor(25, 25, arcade.color.BLUE)
        self.player.center_x = 100
        self.player.center_y = 100

        self.walls = arcade.SpriteList()
        # Sol très long
        for x in range(0, 3001, 64):
            w = arcade.SpriteSolidColor(64, 20, arcade.color.DARK_GREEN)
            w.center_x = x; w.center_y = 32
            self.walls.append(w)

        self.physics = arcade.PhysicsEnginePlatformer(
            self.player, self.walls, gravity_constant=0.8
        )

    def on_key_press(self, key, mods):
        if key == arcade.key.LEFT: self.player.change_x = -5
        elif key == arcade.key.RIGHT: self.player.change_x = 5
        elif key == arcade.key.UP and self.physics.can_jump():
            self.player.change_y = 14

    def on_key_release(self, key, mods):
        if key in (arcade.key.LEFT, arcade.key.RIGHT):
            self.player.change_x = 0

    def on_update(self, dt):
        self.physics.update()
        # Centrer la caméra sur le joueur
        self.camera.position = (
            self.player.center_x - 400,
            0
        )

    def on_draw(self):
        self.clear()
        self.camera.use()
        self.walls.draw()
        self.player.draw()

        # HUD (sans scroll)
        self.gui_camera.use()
        progress = self.player.center_x / 3000
        arcade.draw_rectangle_filled(400, 580, 760, 10, arcade.color.DARK_GRAY)
        arcade.draw_rectangle_filled(20 + progress*760, 580, 10, 10, arcade.color.RED)

# game = ScrollingGame()
# arcade.run()
print('Scrolling game défini')

### Ressources pour aller plus loin

- **Arcade** : [api.arcade.academy](https://api.arcade.academy) -- documentation officielle
- **Pygame-CE** : [pyga.me](https://pyga.me) -- fork communautaire maintenu
- **Kenney.nl** : assets gratuits (sprites, tilesets, sons)
- **Tiled** : éditeur de tilemaps compatible Arcade
- **PyInstaller** : pour distribuer votre jeu en `.exe`

### Exercice bonus : tilemap avec Tiled

Arcade peut charger des tilemaps créées avec l'éditeur **Tiled** (format TMX).
Voici le code de chargement type :

In [ ]:
import arcade

class TiledGame(arcade.Window):
    """Exemple de chargement de tilemap Tiled."""
    def __init__(self):
        super().__init__(800, 600, 'Tiled Map')

        # Chargement de la tilemap
        # tile_map = arcade.load_tilemap('map.tmx', scaling=1.0)
        # self.scene = arcade.Scene.from_tilemap(tile_map)

        # Alternative : tilemap programmatique
        self.scene = arcade.Scene()
        walls = arcade.SpriteList()
        for x in range(0, 1601, 32):
            for y in range(0, 65, 32):
                w = arcade.SpriteSolidColor(32, 32, arcade.color.DARK_GREEN)
                w.center_x = x; w.center_y = y
                walls.append(w)
        self.scene.add_sprite_list('walls', sprite_list=walls)

        # Joueur
        self.player = arcade.SpriteSolidColor(24, 24, arcade.color.BLUE)
        self.player.center_x = 100; self.player.center_y = 100
        self.scene.add_sprite('player', self.player)

        self.physics = arcade.PhysicsEnginePlatformer(
            self.player,
            walls=self.scene.get_sprite_list('walls'),
            gravity_constant=0.8
        )

    def on_key_press(self, key, mods):
        if key == arcade.key.LEFT: self.player.change_x = -5
        elif key == arcade.key.RIGHT: self.player.change_x = 5
        elif key == arcade.key.UP and self.physics.can_jump():
            self.player.change_y = 14

    def on_key_release(self, key, mods):
        if key in (arcade.key.LEFT, arcade.key.RIGHT):
            self.player.change_x = 0

    def on_update(self, dt):
        self.physics.update()

    def on_draw(self):
        self.clear()
        self.scene.draw()

# game = TiledGame()
# arcade.run()
print('Tiled game défini')

### Astuce : structure d'un projet Arcade

```
mon_jeu_arcade/
  main.py           # point d'entrée
  constants.py      # constantes du jeu
  views/
    menu.py         # MenuView
    game.py         # GameView
    game_over.py    # GameOverView
  entities/
    player.py       # Sprite joueur
    enemies.py      # Sprites ennemis
  maps/
    level_01.tmx    # Tilemap Tiled
  resources/
    images/
    sounds/
```

### Exercice : audio et effets

Arcade intègre la gestion audio de manière simple :

In [ ]:
import arcade

# Charger et jouer un son
# coin_sound = arcade.load_sound('coin.wav')
# arcade.play_sound(coin_sound, volume=0.5)

# Musique de fond
# music = arcade.Sound('background.mp3', streaming=True)
# music_player = music.play(volume=0.3, loop=True)

# Arrêter la musique
# music.stop(music_player)

print('API audio Arcade documentée')
print('Note : arcade.Sound supporte .wav, .ogg, .mp3')

### Pour aller plus loin avec Arcade

- **Hitboxes** : `arcade.hitbox.PymunkHitBoxAlgorithm` pour des hitboxes précises
- **Pymunk** : intégration avec le moteur physique Pymunk pour la physique réaliste
- **Shaders** : support OpenGL pour les effets visuels avancés
- **GUI** : `arcade.gui` pour les boutons, sliders, etc. dans le jeu
- **Networking** : possibilité de jeu multijoueur via websockets

---

## Ressources externes

- [Arcade — documentation officielle](https://api.arcade.academy/)
- [Arcade tutorials](https://api.arcade.academy/en/latest/tutorials/index.html)
- [Arcade examples](https://api.arcade.academy/en/latest/examples/index.html)